<a href="https://colab.research.google.com/github/Dev-Maidul/Assignment-1/blob/main/ML_Assignment_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML Assignment 02
### Dataset: House Price Prediction Dataset
### Total Marks: 100

---

## Exam Instructions:
1. প্রথমে নিচের cell এ নিজের **নাম** এবং কোর্সে registration করা **ইমেইল** দিবে
2. Question wise numbering করে Text cell রাখবে এবং এর নিচে Code cell থাকবে, চেষ্টা করবে একটি code cell এ একটি question উত্তর দেওয়ার
3. Google Colab এর মধ্যে কোডগুলো করবে
4. এবং সেই ফাইলটি **'Anyone with the link' & 'View' Access** দিয়ে ফাইলটির Shareable Link টি সাবমিট করবে

---

**Question Dataset Link:** https://www.kaggle.com/datasets/prokshitha/home-value-insights

## Student Information

In [68]:
# Fill in your information
name = "Md Maidul Islam"           # Write your full name here
email = "maidulislammanik8991@gmail.com"          # Write your registered email here

print(f"Name  : {name}")
print(f"Email : {email}")

Name  : Md Maidul Islam
Email : maidulislammanik8991@gmail.com


In [95]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

In [101]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import SGDRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_squared_error,
    r2_score
)

---
## Question 1 (10 Marks)

Load the House Price dataset and display:
- Dataset shape
- First 10 rows
- 5 random samples

In [70]:
# Question 1
df=pd.read_csv("/content/house_price_regression_dataset.csv")
display(df.shape)
display(df.head(10))



(1000, 8)

,Square_Footage,Num_Bedrooms,Num_Bathrooms,Year_Built,Lot_Size,Garage_Size,Neighborhood_Quality,House_Price
0,1360,2,1,1981,0.599637,0,5,2.623829e+05
1,4272,3,3,2016,4.753014,1,6,9.852609e+05
2,3592,1,2,2016,3.634823,0,9,7.779774e+05
3,966,1,2,1977,2.730667,1,8,2.296989e+05
4,4926,2,1,1993,4.699073,0,8,1.041741e+06
5,3944,5,3,1990,2.475930,2,8,8.797970e+05
6,3671,1,2,2012,4.911960,0,1,8.144279e+05
7,3419,1,1,1972,2.805281,1,1,7.034131e+05
8,630,3,3,1997,1.014286,1,8,1.738750e+05
9,2185,4,2,1981,3.941604,2,5,5.041765e+05


In [71]:
display(df.sample(5))

,Square_Footage,Num_Bedrooms,Num_Bathrooms,Year_Built,Lot_Size,Garage_Size,Neighborhood_Quality,House_Price
154,3277,4,3,1996,4.701070,1,10,752466.674667
876,3868,1,3,2014,4.268889,0,4,864309.032358
755,2560,4,2,2015,3.802574,0,2,622100.189502
195,4143,1,2,1984,2.723665,1,8,845141.616664
326,2182,2,1,1967,0.571204,1,2,408013.579062


---
## Question 2 (10 Marks)

Handle missing values and perform feature engineering:
- Impute missing numerical values using `SimpleImputer` with mean strategy
- Impute missing categorical values using most frequent strategy
- Drop columns with more than 50% missing values
- Perform train-test split with `test_size=0.2` and `random_state=42`

Display the shape of final train and test sets.

In [72]:
missing_percentage = df.isnull().mean() * 100

columns_to_drop = missing_percentage[
    missing_percentage > 50
].index

df = df.drop(columns=columns_to_drop)

print("Dropped Columns:")
print(list(columns_to_drop))

Dropped Columns:
[]


In [73]:
numerical_columns = df.select_dtypes(
    include=["int64", "float64"]
).columns

categorical_columns = df.select_dtypes(
    include=["object"]
).columns

In [74]:
num_imputer=SimpleImputer(
    strategy="mean"
)
df[numerical_columns]=num_imputer.fit_transform(
    df[numerical_columns]
)

In [75]:
# print("Numerical Columns:", list(numerical_columns))
# print("Categorical Columns:", list(categorical_columns))

In [76]:
cat_imputer = SimpleImputer(strategy="most_frequent")

if len(categorical_columns) > 0:
    df[categorical_columns] = cat_imputer.fit_transform(
        df[categorical_columns]
    )

In [77]:
X=df.drop("House_Price",axis=1)
y=df["House_Price"]

In [78]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [79]:
print("X_train Shape:", X_train.shape)
print("X_test Shape :", X_test.shape)

print("y_train Shape:", y_train.shape)
print("y_test Shape :", y_test.shape)

X_train Shape: (800, 7)
X_test Shape : (200, 7)
y_train Shape: (800,)
y_test Shape : (200,)


---
## Question 3 (20 Marks)

Implement **Simple Linear Regression** using **only NumPy** (no Scikit-Learn allowed):
- Compute slope (`m`) and intercept (`c`) using the Batch Gradient Descent
- Predict values for the test set
- Print the learned `m` and `c` values

Use `Square_Footage` as feature (X) and `House_Price` as target (y).

In [80]:
#Question 3
X=df["Square_Footage"].values
y=df["House_Price"].values

In [81]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [82]:
def make_prediction(x,m,c):
  return m*x+c

In [83]:
def compute_cost(X,y,m,c):
  n=len(X)
  predictions=make_prediction(X,m,c)
  cost=np.sum((predictions-y)**2 / (2*n))
  return cost

In [84]:
def calculate_gradient(X, y, m, c):

    n = len(X)

    dm = 0
    dc = 0

    for i in range(n):

        prediction = m * X[i] + c

        error = prediction - y[i]

        dm += error * X[i]

        dc += error

    dm = dm / n

    dc = dc / n

    return dm, dc

In [85]:
def gradient_descent(
    X,
    y,
    m,
    c,
    learning_rate,
    iterations
):

    cost_history = []

    for i in range(iterations):

        dm, dc = calculate_gradient(
            X,
            y,
            m,
            c
        )

        m = m - learning_rate * dm

        c = c - learning_rate * dc

        cost = compute_cost(
            X,
            y,
            m,
            c
        )

        cost_history.append(cost)

        if i % 100 == 0:
            print(
                f"Iteration {i}: Cost = {cost}"
            )

    return m, c, cost_history

In [86]:
m = 0
c = 0

In [87]:
m, c, cost_history = gradient_descent(
    X_train,
    y_train,
    m,
    c,
    learning_rate=0.00000001,
    iterations=1000
)

Iteration 0: Cost = 183179374986.2054
Iteration 100: Cost = 806479753.288814
Iteration 200: Cost = 806479270.6496542
Iteration 300: Cost = 806479190.3450087
Iteration 400: Cost = 806479110.04039
Iteration 500: Cost = 806479029.7357986
Iteration 600: Cost = 806478949.4312333
Iteration 700: Cost = 806478869.1266949
Iteration 800: Cost = 806478788.8221825
Iteration 900: Cost = 806478708.5176969


In [92]:
print("Learned Slope (m):", m)

print("Learned Intercept (c):", c)

Learned Slope (m): 216.64402648904004
Learned Intercept (c): 0.15388085589545406


In [93]:
y_pred = make_prediction(
    X_test,
    m,
    c
)

In [94]:
for i in range(10):

    print(
        f"Actual: {y_test[i]:.2f}, "
        f"Predicted: {y_pred[i]:.2f}"
    )

Actual: 901000.49, Predicted: 869175.99
Actual: 494537.51, Predicted: 500447.86
Actual: 949404.20, Predicted: 1019960.23
Actual: 1040389.05, Predicted: 1068488.49
Actual: 794010.02, Predicted: 789884.27
Actual: 724033.56, Predicted: 776885.63
Actual: 998439.24, Predicted: 1004795.15
Actual: 909713.44, Predicted: 894090.05
Actual: 792681.52, Predicted: 819131.22
Actual: 947490.78, Predicted: 919220.76


---
## Question 4 (10 Marks)

Build a **ColumnTransformer** that applies:
- `StandardScaler` on numerical columns: `Square_Footage`, `Num_Bedrooms`, `Num_Bathrooms`
- `OneHotEncoder` on categorical column: `Neighborhood_Quality`



In [96]:
#Question 4
numerical_features = [
    "Square_Footage",
    "Num_Bedrooms",
    "Num_Bathrooms"
]

In [97]:
categorical_features = [
    "Neighborhood_Quality"
]

In [98]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(),
            categorical_features
        )
    ]
)

In [100]:
X_transformed = preprocessor.fit_transform(df)
print(X_transformed.shape)

(1000, 13)


## Question 5 (20 Marks)

Build a complete **Pipeline** using Scikit-Learn that includes:
- The `ColumnTransformer`
- `SGDRegressor` as the final estimator
- Train the pipeline and evaluate using RMSE and R² score
- Print predicted vs actual values for the first 10 test samples

In [102]:
# Question 5
X = df.drop(
    "House_Price",
    axis=1
)

y = df["House_Price"]

In [103]:
X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,random_state=42
)

In [104]:
numerical_features = [
    "Square_Footage",
    "Num_Bedrooms",
    "Num_Bathrooms"
]

categorical_features = [
    "Neighborhood_Quality"
]

In [107]:
preprocessor=ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features

         ),
        (
            "cat",
            OneHotEncoder(),
            categorical_features
        )
    ]
)

In [108]:
pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),

        (
            "model",
            SGDRegressor(
                random_state=42
            )
        )
    ]
)

In [109]:
pipeline.fit(
    X_train,
    y_train
)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['Square_Footage',
                                                   'Num_Bedrooms',
                                                   'Num_Bathrooms']),
                                                 ('cat', OneHotEncoder(),
                                                  ['Neighborhood_Quality'])])),
                ('model', SGDRegressor(random_state=42))])

In [110]:
y_pred = pipeline.predict(
    X_test
)

In [111]:
rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred
    )
)

In [112]:
r2 = r2_score(
    y_test,
    y_pred
)

In [113]:
print("RMSE :", rmse)

print("R² Score :", r2)

RMSE : 29096.460915896132
R² Score : 0.9868659781432683


In [114]:
print(
    "\nFirst 10 Predictions:\n"
)

for i in range(10):

    print(
        f"Actual: {y_test.iloc[i]:.2f}"
        f" | Predicted: {y_pred[i]:.2f}"
    )


First 10 Predictions:

Actual: 901000.49 | Predicted: 851369.91
Actual: 494537.51 | Predicted: 507342.82
Actual: 949404.20 | Predicted: 985831.68
Actual: 1040389.05 | Predicted: 1022464.74
Actual: 794010.02 | Predicted: 759809.08
Actual: 724033.56 | Predicted: 765675.68
Actual: 998439.24 | Predicted: 1004527.78
Actual: 909713.44 | Predicted: 903546.05
Actual: 792681.52 | Predicted: 797193.08
Actual: 947490.78 | Predicted: 888626.74


---
## Question 6 (20 Marks)

Implement **Multiple Linear Regression** using **Scikit-Learn**:
- The `ColumnTransformer`
- `LinearRegression` as the final estimator
- Train the pipeline and evaluate using RMSE and R² score
- Print predicted vs actual values for the first 10 test samples

In [90]:
# Question 6


---
## Question 7 (10 Marks) (You have to explore the topic and use the equation via Numpy)
### Dont use LLMs , You can use Documentation

Implement **Multiple Linear Regression** using **only NumPy**:
- Pick random 100 datas from the dataset
- Use the Normal Equation: `θ = (XᵀX)⁻¹ Xᵀy`
- Use `Square_Footage`, `Num_Bedrooms`, and `Num_Bathrooms` as features
- Print the learned coefficients (θ values)

In [91]:
# Question 7
